# Урок 17. Задачи машинного обучения: признаки, цель, классификация и регрессия

**Дата:** 14.01.2027 · **60 минут** · 8–9 класс  
**Модуль 4.** Машинное обучение (уроки 17–25)  
**Опирается на:** урок 1 (правила и ML), уроки 10–12 (pandas, разведка данных).

Начинаем главный модуль курса. Но прежде чем обучать модели, нужно научиться **ставить задачу**: что именно предсказываем, по каким данным и как поймём, что получилось. Ошибка в постановке не лечится никаким алгоритмом — поэтому сегодня без моделей, зато с инженерным мышлением.

## Цели урока

К концу занятия ты сможешь:

1. Называть в таблице **объекты**, **признаки (X)** и **цель (y)**.
2. Отличать задачу **классификации** от задачи **регрессии**.
3. Проверять признак на **утечку**: известен ли он в момент предсказания?
4. Выделять `X` и `y` из таблицы pandas.
5. Заполнять «паспорт задачи» для ML-проекта.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–5 | Вспоминаем: обучение на примерах |
| 5–20 | Объекты, признаки, цель; обучение с учителем |
| 20–30 | Классификация и регрессия |
| 30–40 | Утечка: признак из будущего |
| 40–55 | Практика: паспорта задач |
| 55–60 | Итог, подготовка к СР №2 |

## 1. Вспоминаем: обучение на примерах

На уроке 1 ты сам искал правило «если часов не меньше …, то сдаст», глядя на примеры с ответами. Машинное обучение делает то же самое, только:

- примеров тысячи, а не восемь;
- столбцов много, а не один;
- правило бывает гораздо сложнее одного порога.

Найденное правило называют **моделью**, а процесс поиска — **обучением**.

## 2. Объекты, признаки и цель

Посмотри на таблицу учеников:

| часы | посещаемость | кружок | **сдал** |
|-----:|-------------:|--------|:--------:|
| 8 | 90 | Python | да |
| 3 | 55 | Роботы | нет |
| 10 | 95 | Python | да |

- **Объект** (пример) — одна **строка**: один ученик.
- **Признаки** — столбцы, по которым делаем предсказание: часы, посещаемость, кружок. Их вместе обозначают **X**.
- **Цель** (целевая переменная) — столбец, который хотим предсказать для **новых** объектов: сдал. Её обозначают **y**.

> **Аналогия.** Врач (модель) смотрит на симптомы (признаки X) и ставит диагноз (цель y). Учился он на историях болезней, где диагноз уже был известен.

### Обучение с учителем

Если в данных для обучения **цель известна** (у каждого ученика записано, сдал он или нет), это **обучение с учителем**. «Учитель» — это правильные ответы в примерах. Их ещё называют **разметкой**. Весь наш модуль — про обучение с учителем.

Разметку часто делают люди: например, вручную отмечают тысячи писем «спам / не спам». Это долгая и дорогая работа, и от её качества зависит качество модели.

## 3. Классификация и регрессия

Тип задачи определяется тем, **какая цель**:

| | Классификация | Регрессия |
|--|---------------|-----------|
| Цель y | **класс** (метка из списка) | **число** на шкале |
| Вопрос | «какой?», «кто?», «да или нет?» | «сколько?» |
| Примеры | сдаст / не сдаст; спам / не спам; кошка / собака / птица | балл 0–100; цена квартиры; время в пути |

Если классов два (да / нет) — классификация **бинарная**, если больше — **многоклассовая**.

От типа задачи зависит, какие модели брать и как измерять качество. Поэтому тип называют **до** начала работы.

### Одни данные — две задачи

На одной и той же таблице учеников можно поставить обе задачи:

- **классификация:** y = `сдал` (да / нет);
- **регрессия:** y = `балл` (число от 0 до 100).

Это **разные** задачи — с разными целями, разными способами оценки и разной пользой. Выбор зависит от того, что нужно пользователю: «кому срочно помочь» — классификация, «сколько баллов ждать» — регрессия.

## 4. Выделяем X и y в pandas

Возьмём учебные данные (генератор с уроков 12–13) и добавим столбцы `id` и `имя`, как в настоящей таблице.

In [ ]:
import random
import pandas as pd

def make_students(n, seed):
    """Сгенерировать учебную таблицу учеников (как на уроке 12)."""
    random.seed(seed)
    rows = []
    for i in range(n):
        hours = random.randint(1, 14)
        attendance = min(100, max(20, round(50 + 3 * hours + random.gauss(0, 10))))
        score = min(100, max(0, round(20 + 4 * hours + 0.3 * attendance + random.gauss(0, 8))))
        rows.append({
            "id": i + 1,
            "имя": random.choice(["Аня", "Боря", "Вика", "Глеб", "Даша", "Егор", "Женя", "Зоя"]),
            "класс": random.choice([8, 9]),
            "кружок": random.choice(["Python", "Роботы", "Игры"]),
            "часы": hours,
            "посещаемость": attendance,
            "сон": round(random.uniform(6, 9.5), 1),
            "балл": score,
        })
    return pd.DataFrame(rows)

df = make_students(60, seed=12)
df["сдал"] = df["балл"] >= 60
df.head()

Поставим задачу классификации: **заранее, в середине года**, предсказать, сдаст ли ученик итоговую работу, чтобы вовремя предложить помощь.

In [ ]:
feature_cols = ["часы", "посещаемость", "класс"]
target_col = "сдал"

X = df[feature_cols]
y = df[target_col]

print("X:", X.shape, "— строк и признаков")
print("y:", y.shape, "— одна цель для каждой строки")
print()
print(y.value_counts())

Строки `X` и `y` идут **в одном порядке**: первая строка `X` — признаки первого ученика, первое значение `y` — его результат. Это важно сохранять всегда.

## 5. Какие столбцы **нельзя** брать в признаки

### Проверка «момент предсказания»

Главный вопрос к каждому признаку:

> **Будет ли это значение известно в тот момент, когда модель должна дать ответ?**

Мы предсказываем в середине года. Часы и посещаемость к этому моменту известны. А `балл` за итоговую работу — **нет**: работа ещё не написана. Если взять его в признаки, модель будет «подглядывать в ответ». Это называется **утечкой** (*data leakage*).

Посмотрим, что даёт «модель» с утечкой:

In [ ]:
cheat_guess = df["балл"] >= 60
share_correct = (cheat_guess == df["сдал"]).mean()
print("Доля верных ответов «модели», которая знает балл:", share_correct)

100% — потому что `сдал` **вычислен** из балла. На старых данных такая модель «идеальна», а в жизни бесполезна: в середине года балла ещё нет. Подозрительно высокое качество — первый признак утечки.

### Идентификаторы и имена

`id` и `имя` — тоже **не признаки**. Модель не должна запоминать «Вика всегда сдаёт». Нужны свойства, которые **обобщаются** — то есть помогают предсказывать и для новых учеников, которых модель не видела: часы, посещаемость. К тому же использовать имена в модели — неэтично: люди с одинаковым именем не похожи по успеваемости.

### Примеры утечек

| Задача | Опасный признак | Почему нельзя |
|--------|-----------------|---------------|
| Сдаст ли ученик зачёт | итоговый балл | известен только после зачёта |
| Уйдёт ли клиент из сервиса | дата ухода | появляется только после ухода |
| Болен ли пациент | назначенное лекарство | назначают после диагноза |
| Спам ли письмо | «пользователь пометил как спам» | это и есть ответ |

## 6. Признаки бывают разными

| Вид | Пример | Как хранится |
|-----|--------|--------------|
| числовые | часы, посещаемость, сон | `int`, `float` |
| категориальные | кружок, класс | текст или число-код |
| логические | занимается спортом | `True / False` |

Модели, которые мы начнём обучать на уроке 19, умеют обрабатывать только **числа**. Текстовые категории («Python», «Роботы») перед обучением превращают в числа — этим займёмся на уроке 20. Пока в примерах берём числовые признаки.

## 7. Паспорт задачи

Каждый ML-проект начинают с паспорта. Он же понадобится в итоговом проекте.

| Пункт | Пример |
|-------|--------|
| **Пользователь** | куратор кружков |
| **Зачем** | заранее увидеть, кому нужна помощь перед итоговой работой |
| **Момент предсказания** | середина года |
| **Цель y** | `сдал` (да / нет) |
| **Тип задачи** | бинарная классификация |
| **Признаки X** | часы, посещаемость, класс |
| **Не берём** | `балл` (утечка), `id` и `имя` (не обобщаются) |
| **Как поймём, что хорошо** | модель находит большинство тех, кто не сдаст |
| **Риски** | данных мало; ошибка «не заметили, кому нужна помощь» |

## 8. Практика

### Практика 1. Классификация или регрессия? (7 минут)

Поставь **К** или **Р**. Для каждой задачи назови цель y.

| № | Задача | К / Р | Цель y |
|--:|--------|:-----:|--------|
| 1 | Определить, спам ли письмо | | |
| 2 | Предсказать температуру воздуха на завтра | | |
| 3 | Узнать, кошка или собака на фото | | |
| 4 | Оценить цену подержанного велосипеда | | |
| 5 | Предсказать, купит ли пользователь товар | | |
| 6 | Сколько лайков наберёт пост | | |
| 7 | Распознать рукописную цифру от 0 до 9 | | |
| 8 | Сколько минут займёт дорога до школы | | |
| 9 | Определить настроение отзыва: плохой / нейтральный / хороший | | |

<details>
<summary>Ответы</summary>

1 — К (спам / не спам), 2 — Р (градусы), 3 — К (кошка / собака), 4 — Р (рубли), 5 — К (купит / нет), 6 — Р (число лайков), 7 — К (10 классов), 8 — Р (минуты), 9 — К (3 класса).

Задача 6 спорная: если важно «наберёт больше 100 лайков или нет», это уже классификация. **Тип задачи зависит от того, как мы поставили цель**.
</details>

### Практика 2. Найди утечку (6 минут)

Задача: **в начале смены** в летнем лагере предсказать, понравится ли ребёнку смена (да / нет). Какие столбцы можно брать в признаки, а какие нет — и почему?

- возраст;
- был ли в лагере раньше;
- оценка смены в анкете в последний день;
- сколько раз звонил родителям за смену;
- выбранный кружок;
- номер путёвки.

**Можно:** …

**Нельзя (и почему):** …

<details>
<summary>Ответ</summary>

Можно: возраст, был ли раньше, выбранный кружок — известны в начале смены.

Нельзя: оценка в последний день — это почти и есть ответ, она появится только в конце (утечка); число звонков за смену — становится известным только к концу смены (утечка); номер путёвки — идентификатор, не обобщается.
</details>

### Практика 3. Регрессия на тех же данных (5 минут)

Поставь задачу **регрессии** на таблице `df`: цель — `балл`. Выдели `X` и `y`. Какие столбцы нельзя брать в признаки на этот раз?

In [ ]:
# Практика 3

<details>
<summary>Решение</summary>

```python
X_reg = df[["часы", "посещаемость", "класс"]]
y_reg = df["балл"]
print(X_reg.shape, y_reg.shape)
```
Нельзя брать `сдал`: он вычислен из балла — это тоже утечка. И, как всегда, `id` и `имя`.
</details>

### Практика 4. Паспорт своей задачи (10 минут)

Придумай задачу из своей жизни (спорт, игры, учёба, кино, погода) и заполни паспорт. Это черновик идеи для итогового проекта.

| Пункт | Мой ответ |
|-------|-----------|
| Пользователь | |
| Зачем | |
| Момент предсказания | |
| Цель y | |
| Тип задачи | |
| Признаки X | |
| Не берём (и почему) | |
| Как поймём, что хорошо | |
| Риски | |

## Работа с ИИ-помощником

- «Проверь, нет ли утечки в моём списке признаков: задача …, признаки …»
- «Приведи ещё три примера задач регрессии из жизни школьника.»
- Плохо: «Придумай мне проект и заполни паспорт». Паспорт — это твоё понимание задачи.

## Мини-глоссарий урока 17

| Термин | Смысл |
|--------|-------|
| **Объект (пример)** | одна строка таблицы |
| **Признаки, X** | столбцы, по которым предсказываем |
| **Цель, y** | столбец, который предсказываем |
| **Обучение с учителем** | обучение на примерах с известными ответами |
| **Разметка** | правильные ответы в обучающих данных |
| **Классификация** | цель — класс из списка |
| **Регрессия** | цель — число |
| **Утечка** | признак, который «знает ответ» или недоступен в момент предсказания |

## Итог урока

1. Строка — объект, признаки — X, цель — y; X и y всегда идут в одном порядке.
2. Цель-класс → классификация, цель-число → регрессия.
3. Каждый признак проверяем вопросом «известен ли он в момент предсказания?».

## Домашнее задание

Подготовься к самостоятельной работе №2 (21.01): она проверяет **модуль данных** (уроки 10–14) и сегодняшнюю постановку задачи.

- [ ] паспорт таблицы: `shape`, `head`, `info`, `describe`;
- [ ] очистка: пропуски, дубликаты, типы, выбросы + журнал решений;
- [ ] разведка: `value_counts`, `groupby` с `count`, корреляция;
- [ ] график с подписями и выводом;
- [ ] выбор `X` и `y` и проверка на утечку.

Повтори урок 14 — он устроен почти как СР №2.

## Самопроверка

1. Что такое объект, признак и цель? Покажи на примере таблицы учеников.
2. Чем классификация отличается от регрессии?
3. Почему `балл` нельзя брать в признаки, если предсказываем `сдал`?
4. Почему `id` не признак?
5. Что такое разметка и кто её делает?

**Следующий урок (21.01.2027):** самостоятельная работа №2 — анализ данных.